# Monte Carlo Sampling

## Learning Objectives
1. Understand how Monte Carlo integration approximates expectations and why error scales as 1/sqrt(N)
2. Implement importance sampling to estimate rare-event probabilities efficiently
3. Apply variance reduction techniques (antithetic variates, control variates) to reduce sample requirements
4. Compare standard MC, importance sampling, and antithetic variates on real estimation problems

In [ ]:
# Cell 2: Imports and reproducibility setup
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

np.random.seed(42)

# Matplotlib style for clean plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully')
print(f'NumPy version: {np.__version__}')

## Level 1: Basic Monte Carlo Integration

We estimate pi using the unit circle and demonstrate that error shrinks as 1/sqrt(N).

In [ ]:
# Cell 4: Basic MC integration - estimating pi
# Core formula: E[f(X)] ≈ (1/N) * sum(f(x_i)) where x_i ~ p(x)
# For pi: sample (x,y) ~ Uniform[-1,1]^2, f = 1 if x^2+y^2 <= 1
# E[f] = P(point in circle) = pi/4, so pi ≈ 4 * mean(f)

def estimate_pi_mc(n_samples: int, seed: int = 42) -> tuple:
    """Estimate pi via MC integration using the unit circle.
    
    Args:
        n_samples: Number of random samples to draw
        seed: Random seed for reproducibility
    
    Returns:
        Tuple of (pi_estimate, standard_error)
    """
    rng = np.random.default_rng(seed)
    # Sample uniformly in [-1, 1]^2
    x = rng.uniform(-1, 1, n_samples)
    y = rng.uniform(-1, 1, n_samples)
    # Indicator: 1 if inside unit circle
    inside = (x**2 + y**2 <= 1.0).astype(float)
    # E[inside] = pi/4, so pi ≈ 4 * mean(inside)
    pi_estimate = 4.0 * np.mean(inside)
    # Std error of the mean = std(f) / sqrt(N)
    std_error = 4.0 * np.std(inside) / np.sqrt(n_samples)
    return pi_estimate, std_error

# Show convergence across many sample sizes
sample_sizes = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
estimates = []
errors = []

print(f'{'N':>10}  {'pi_estimate':>12}  {'std_error':>10}  {'true_error':>12}')
print('-' * 52)
for n in sample_sizes:
    est, err = estimate_pi_mc(n, seed=42)
    true_err = abs(est - np.pi)
    estimates.append(est)
    errors.append(err)
    print(f'{n:>10,}  {est:>12.6f}  {err:>10.6f}  {true_err:>12.6f}')

print(f'\nTrue pi = {np.pi:.6f}')

# Show that error scales as 1/sqrt(N) on log-log plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Running estimate with confidence band
rng = np.random.default_rng(0)
x_run = rng.uniform(-1, 1, 100_000)
y_run = rng.uniform(-1, 1, 100_000)
inside_run = (x_run**2 + y_run**2 <= 1.0).astype(float)
running_mean = 4.0 * np.cumsum(inside_run) / np.arange(1, len(inside_run) + 1)
ns = np.arange(1, len(inside_run) + 1)
running_std = 4.0 * np.sqrt(inside_run.var() / ns)

axes[0].plot(ns[::100], running_mean[::100], color='navy', lw=1.5, label='MC estimate')
axes[0].axhline(np.pi, color='red', lw=1.5, ls='--', label=f'True pi = {np.pi:.4f}')
axes[0].fill_between(ns[::100],
                     running_mean[::100] - 2 * running_std[::100],
                     running_mean[::100] + 2 * running_std[::100],
                     alpha=0.2, color='navy', label='95% CI')
axes[0].set_xscale('log')
axes[0].set_xlabel('N samples')
axes[0].set_ylabel('pi estimate')
axes[0].set_title('MC convergence to pi')
axes[0].legend()

# Right: log-log error vs N shows slope ≈ -0.5
ns_arr = np.array(sample_sizes)
errors_arr = np.array([abs(e - np.pi) for e in estimates])
theoretical = 1.0 / np.sqrt(ns_arr)  # 1/sqrt(N) reference line

axes[1].loglog(ns_arr, errors_arr, 'o-', color='navy', label='|estimate - pi|')
axes[1].loglog(ns_arr, theoretical * errors_arr[0] * np.sqrt(ns_arr[0]),
               '--', color='red', label='O(1/sqrt(N)) reference')
axes[1].set_xlabel('N samples')
axes[1].set_ylabel('Absolute error')
axes[1].set_title('Error vs N (log-log) — slope -0.5')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Level 1: Basic Monte Carlo Integration', y=1.01, fontsize=13, fontweight='bold')
plt.savefig('mc_level1.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 1 complete.')

## Level 2: Importance Sampling for Rare Events

Standard MC is catastrophically inefficient for P(X > 4) for X ~ N(0,1) because the event is rare (~3.2e-5). Importance sampling uses a shifted proposal to focus samples in the rare region.

In [ ]:
# Cell 6: Importance sampling — estimate P(X > 4) for X ~ N(0,1)
# True value: P(X > 4) ≈ 3.167e-5
# Standard MC needs ~1/p ≈ 31,000 samples just to see one event
# IS solution: sample from N(4, 1) which concentrates mass near threshold
# IS estimator: E_p[f(X)] = E_q[f(X) * p(X)/q(X)]
# Weight w_i = p(x_i) / q(x_i) for each sample x_i ~ q

THRESHOLD = 4.0  # estimate P(X > threshold)
TRUE_PROB = 1 - stats.norm.cdf(THRESHOLD)  # ground truth

print(f'Target: P(X > {THRESHOLD}) = {TRUE_PROB:.2e}')
print(f'Avg samples to see one event with naive MC: {1/TRUE_PROB:.0f}\n')


def naive_mc_tail(threshold: float, n_samples: int, rng) -> tuple:
    """Estimate P(X > threshold) with naive MC sampling from N(0,1).
    
    Returns (estimate, std_error, effective_sample_size)
    """
    x = rng.normal(0, 1, n_samples)
    indicators = (x > threshold).astype(float)
    estimate = indicators.mean()
    std_error = indicators.std() / np.sqrt(n_samples)
    return estimate, std_error, n_samples


def importance_sampling_tail(threshold: float, n_samples: int, rng,
                             proposal_mean: float = 4.5) -> tuple:
    """Estimate P(X > threshold) using importance sampling.
    
    Proposal: q(x) = N(proposal_mean, 1). Weights: p(x)/q(x).
    f(x) = 1 if x > threshold, else 0.
    Estimator: mean(f(x_i) * w_i) where w_i = p(x_i)/q(x_i)
    
    Returns (estimate, std_error, effective_sample_size)
    """
    # Sample from the proposal distribution
    x = rng.normal(proposal_mean, 1.0, n_samples)
    # Compute importance weights w = p(x)/q(x)
    log_p = stats.norm.logpdf(x, loc=0, scale=1)
    log_q = stats.norm.logpdf(x, loc=proposal_mean, scale=1)
    log_weights = log_p - log_q
    weights = np.exp(log_weights)
    # f(x) = indicator(x > threshold)
    f_x = (x > threshold).astype(float)
    # IS estimator: mean(f * w)
    weighted_f = f_x * weights
    estimate = weighted_f.mean()
    std_error = weighted_f.std() / np.sqrt(n_samples)
    # Effective sample size: ESS = (sum w)^2 / sum(w^2)
    ess = (weights.sum())**2 / (weights**2).sum()
    return estimate, std_error, ess


# Compare naive MC vs IS across different sample sizes
n_sizes = [500, 1000, 5000, 10000, 50000]
rng = np.random.default_rng(42)

print(f'{'N':>8}  {'Naive MC':>12}  {'Naive SE':>10}  {'IS Estimate':>13}  {'IS SE':>10}  {'IS ESS':>8}')
print('-' * 72)
naive_ests, is_ests, naive_ses, is_ses = [], [], [], []

for n in n_sizes:
    naive_est, naive_se, _ = naive_mc_tail(THRESHOLD, n, rng)
    is_est, is_se, ess = importance_sampling_tail(THRESHOLD, n, rng, proposal_mean=4.5)
    naive_ests.append(naive_est)
    is_ests.append(is_est)
    naive_ses.append(naive_se)
    is_ses.append(is_se)
    print(f'{n:>8,}  {naive_est:>12.2e}  {naive_se:>10.2e}  '
          f'{is_est:>13.2e}  {is_se:>10.2e}  {ess:>8.1f}')

print(f'\nTrue P(X > 4) = {TRUE_PROB:.4e}')

# Variance reduction ratio at N=50000
ratio = (naive_ses[-1] / is_ses[-1])**2 if is_ses[-1] > 0 else float('inf')
print(f'\nVariance reduction ratio at N=50000: {ratio:.1f}x')
print('IS achieves equivalent accuracy with ~{:.0f}x fewer samples'.format(ratio))

# Visualize weight distribution to check for weight degeneracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot estimates vs true value
n_arr = np.array(n_sizes)
axes[0].loglog(n_arr, np.abs(np.array(naive_ests) - TRUE_PROB) + 1e-10,
               'o-', color='red', label='Naive MC error')
axes[0].loglog(n_arr, np.abs(np.array(is_ests) - TRUE_PROB) + 1e-10,
               's-', color='navy', label='IS error')
axes[0].loglog(n_arr, TRUE_PROB / np.sqrt(n_arr), '--', color='gray',
               label='O(1/sqrt(N)) reference')
axes[0].set_xlabel('N samples')
axes[0].set_ylabel('|estimate - truth|')
axes[0].set_title('Error vs N: Naive MC vs Importance Sampling')
axes[0].legend()

# Weight distribution with N=2000
x_w = rng.normal(4.5, 1.0, 2000)
log_p = stats.norm.logpdf(x_w, 0, 1)
log_q = stats.norm.logpdf(x_w, 4.5, 1)
weights_w = np.exp(log_p - log_q)
axes[1].hist(weights_w, bins=50, color='navy', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Importance weight w = p(x)/q(x)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'IS Weight Distribution (ESS = {(weights_w.sum()**2 / (weights_w**2).sum()):.0f} / 2000)')

plt.tight_layout()
plt.savefig('mc_level2.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 2 complete.')

## Real-World Example 1: Monte Carlo Option Pricing

Price a European call option by simulating stock paths under the Black-Scholes model and verify against the analytic Black-Scholes formula.

In [ ]:
# Cell 8: MC option pricing with Black-Scholes verification
# Stock price follows geometric Brownian motion:
# S_T = S_0 * exp((r - 0.5*sigma^2)*T + sigma*sqrt(T)*Z) where Z ~ N(0,1)
# Call option payoff: max(S_T - K, 0)
# Call price = e^(-rT) * E[max(S_T - K, 0)]

def mc_call_price(S0: float, K: float, r: float, sigma: float,
                  T: float, n_paths: int, rng) -> tuple:
    """Price European call option via Monte Carlo simulation.
    
    Args:
        S0: Initial stock price
        K: Strike price
        r: Risk-free rate (annualized)
        sigma: Volatility (annualized)
        T: Time to maturity (years)
        n_paths: Number of simulated paths
        rng: NumPy random generator
    
    Returns:
        (price_estimate, standard_error)
    """
    # Simulate terminal stock prices under risk-neutral measure
    Z = rng.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    # Payoff of a call option at maturity
    payoffs = np.maximum(S_T - K, 0)
    # Discount back to present value
    discounted = np.exp(-r * T) * payoffs
    price = discounted.mean()
    se = discounted.std() / np.sqrt(n_paths)
    return price, se


def mc_call_antithetic(S0: float, K: float, r: float, sigma: float,
                       T: float, n_paths: int, rng) -> tuple:
    """Price European call with antithetic variates for variance reduction.
    
    Uses pairs (Z, -Z) which are negatively correlated — when one path
    gives high payoff, its antithetic pair tends to give low payoff,
    reducing estimator variance without bias.
    """
    Z = rng.normal(0, 1, n_paths // 2)  # Half the paths
    # Standard paths and their antithetics (-Z)
    S_T_pos = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    S_T_neg = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * (-Z))
    # Average antithetic pair payoffs
    payoffs_pos = np.maximum(S_T_pos - K, 0)
    payoffs_neg = np.maximum(S_T_neg - K, 0)
    paired_payoffs = (payoffs_pos + payoffs_neg) / 2
    discounted = np.exp(-r * T) * paired_payoffs
    price = discounted.mean()
    se = discounted.std() / np.sqrt(len(discounted))
    return price, se


def black_scholes_call(S0, K, r, sigma, T):
    """Analytic Black-Scholes call price for verification."""
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)


# Option parameters
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.20, 1.0
bs_price = black_scholes_call(S0, K, r, sigma, T)
rng = np.random.default_rng(7)

print(f'Option parameters: S0={S0}, K={K}, r={r}, sigma={sigma}, T={T}')
print(f'Black-Scholes analytic price: ${bs_price:.4f}\n')

path_sizes = [1_000, 5_000, 10_000, 50_000]
print(f'{'N paths':>10}  {'MC Price':>10}  {'MC SE':>8}  {'Anti Price':>12}  {'Anti SE':>10}  {'BS Price':>10}')
print('-' * 68)
for n in path_sizes:
    mc_p, mc_se = mc_call_price(S0, K, r, sigma, T, n, rng)
    anti_p, anti_se = mc_call_antithetic(S0, K, r, sigma, T, n, rng)
    print(f'{n:>10,}  ${mc_p:>9.4f}  {mc_se:>8.4f}  ${anti_p:>11.4f}  {anti_se:>10.4f}  ${bs_price:>9.4f}')

print('\nAntithetic SE is typically 30-50% smaller than standard MC SE at same N.')

## Real-World Example 2: Control Variates for Variance Reduction

A control variate c(x) has a known expectation E[c(X)] = mu_c. We estimate E[f(X)] as:
f*(x) = f(x) - beta*(c(x) - mu_c), then beta* = Cov(f,c)/Var(c) minimizes variance.

In [ ]:
# Cell 10: Control variates — reduce MC variance via known analytic quantities
# Goal: estimate E[exp(X)] for X ~ Uniform(0,1) (true value = e - 1 ≈ 1.718)
# Control variate: c(x) = x, with known E[c(X)] = 0.5
# Adjusted estimator: f*(x_i) = f(x_i) - beta*(c(x_i) - 0.5)
# Optimal beta* = Cov(f,c)/Var(c) — can be estimated from pilot samples

TRUE_VALUE = np.e - 1  # E[exp(X)] for X ~ Uniform(0,1)
N_EXPERIMENTS = 500    # Number of independent MC experiments
N_SAMPLES = 1_000      # Samples per experiment

rng = np.random.default_rng(99)


def run_mc_comparison(n_samples: int, rng) -> dict:
    """Run one MC experiment with and without control variate.
    
    Returns dict with naive and CV estimates.
    """
    x = rng.uniform(0, 1, n_samples)
    f_x = np.exp(x)           # Target function: f(x) = e^x
    c_x = x                   # Control variate: c(x) = x, E[c] = 0.5
    mu_c = 0.5                 # Known expectation of control variate

    # Estimate optimal beta from samples (can also derive analytically)
    beta_hat = np.cov(f_x, c_x)[0, 1] / np.var(c_x)

    # Control variate adjusted estimator
    f_adjusted = f_x - beta_hat * (c_x - mu_c)

    return {
        'naive': np.mean(f_x),
        'cv': np.mean(f_adjusted),
        'beta': beta_hat
    }


# Run many independent experiments to measure variance
naive_estimates = []
cv_estimates = []
betas = []

for _ in range(N_EXPERIMENTS):
    result = run_mc_comparison(N_SAMPLES, rng)
    naive_estimates.append(result['naive'])
    cv_estimates.append(result['cv'])
    betas.append(result['beta'])

naive_estimates = np.array(naive_estimates)
cv_estimates = np.array(cv_estimates)

naive_var = np.var(naive_estimates)
cv_var = np.var(cv_estimates)
variance_reduction = naive_var / cv_var

print(f'Target E[exp(X)] = {TRUE_VALUE:.6f}')
print(f'N experiments: {N_EXPERIMENTS}, N samples per experiment: {N_SAMPLES}')
print(f'\nNaive MC:')
print(f'  Mean estimate: {naive_estimates.mean():.6f}')
print(f'  Variance:      {naive_var:.2e}')
print(f'  Std Error:     {np.sqrt(naive_var):.6f}')
print(f'\nControl Variate MC (c(x) = x):')
print(f'  Mean estimate: {cv_estimates.mean():.6f}')
print(f'  Variance:      {cv_var:.2e}')
print(f'  Std Error:     {np.sqrt(cv_var):.6f}')
print(f'  Beta estimate: {np.mean(betas):.4f} (analytic: {(np.e - 1) - 1:.4f} approx)')
print(f'\nVariance reduction factor: {variance_reduction:.2f}x')
print(f'Equivalent to using {variance_reduction:.1f}x more samples with naive MC')

## Real-World Example 3: Bayesian Posterior Estimation + Comparison

Estimate the Bayesian posterior mean for a Beta-Binomial model using Monte Carlo, then compare all three variance reduction approaches on a common benchmark.

In [ ]:
# Cell 12: Bayesian posterior MC + comparison of MC methods

# ---- Part A: Bayesian Beta-Binomial posterior estimation ----
# Prior: theta ~ Beta(alpha0, beta0)
# Likelihood: k successes in n trials => posterior is Beta(alpha0+k, beta0+n-k)
# Analytic posterior mean = (alpha0+k)/(alpha0+beta0+n)
# MC: sample theta from posterior, compute E[g(theta)] for a nonlinear g
# Here g(theta) = log-odds = log(theta/(1-theta))

alpha0, beta0 = 2.0, 2.0  # Prior: weakly informative
k, n = 6, 10               # Observed: 6 successes in 10 trials
alpha_post = alpha0 + k
beta_post = beta0 + n - k

# Analytic posterior mean for theta (conjugate result)
analytic_theta_mean = alpha_post / (alpha_post + beta_post)
print(f'Beta-Binomial Model: prior Beta({alpha0},{beta0}), data k={k}, n={n}')
print(f'Posterior: Beta({alpha_post:.0f}, {beta_post:.0f})')
print(f'Analytic posterior mean E[theta] = {analytic_theta_mean:.4f}')

# MC estimation of E[log-odds] — no closed form for this
rng = np.random.default_rng(55)
n_posterior_samples = 20_000
theta_samples = rng.beta(alpha_post, beta_post, n_posterior_samples)
log_odds_samples = np.log(theta_samples / (1 - theta_samples))
mc_log_odds = np.mean(log_odds_samples)
mc_log_odds_se = np.std(log_odds_samples) / np.sqrt(n_posterior_samples)
print(f'\nMC estimate E[log-odds] = {mc_log_odds:.4f} ± {mc_log_odds_se:.4f}')
print(f'(No closed form — MC gives principled estimate with quantified uncertainty)')

# ---- Part B: Comprehensive comparison of MC variance reduction methods ----
# Benchmark: E[sin(pi*x/2)] for X ~ Uniform(0,1), true = 2/pi ≈ 0.6366
TRUE_SIN = 2 / np.pi
N_COMP = 2000
N_REPS = 300

methods_results = {'Naive MC': [], 'Antithetic': [], 'Control Variate': []}

for _ in range(N_REPS):
    # Naive MC
    x = rng.uniform(0, 1, N_COMP)
    methods_results['Naive MC'].append(np.mean(np.sin(np.pi * x / 2)))

    # Antithetic variates: pair x with (1-x) for uniform — they sum to 1
    # so sin(pi*x/2) and sin(pi*(1-x)/2) = cos(pi*x/2) are correlated
    x_half = rng.uniform(0, 1, N_COMP // 2)
    f_pos = np.sin(np.pi * x_half / 2)
    f_neg = np.sin(np.pi * (1 - x_half) / 2)  # antithetic on [0,1]
    methods_results['Antithetic'].append(np.mean((f_pos + f_neg) / 2))

    # Control variate: c(x) = x, E[c] = 0.5
    x_cv = rng.uniform(0, 1, N_COMP)
    f_cv = np.sin(np.pi * x_cv / 2)
    c_cv = x_cv
    beta_cv = np.cov(f_cv, c_cv)[0, 1] / np.var(c_cv)
    f_adj = f_cv - beta_cv * (c_cv - 0.5)
    methods_results['Control Variate'].append(np.mean(f_adj))

print(f'\n\nBenchmark: E[sin(pi*X/2)] for X ~ Uniform(0,1)')
print(f'True value = 2/pi = {TRUE_SIN:.6f}, N={N_COMP} samples, {N_REPS} reps\n')
print(f'{'Method':<20}  {'Mean':>10}  {'Std Dev':>10}  {'Variance':>12}  {'Var Reduction':>14}')
print('-' * 72)

naive_var_ref = np.var(methods_results['Naive MC'])
for method, estimates in methods_results.items():
    arr = np.array(estimates)
    v = np.var(arr)
    reduction = naive_var_ref / v if v > 0 else float('inf')
    print(f'{method:<20}  {arr.mean():>10.6f}  {arr.std():>10.6f}  '
          f'{v:>12.2e}  {reduction:>12.2f}x')

# Final comparison plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#e74c3c', '#2980b9', '#27ae60']

for ax, (method, estimates), color in zip(axes, methods_results.items(), colors):
    arr = np.array(estimates)
    ax.hist(arr, bins=40, color=color, alpha=0.75, edgecolor='white', density=True)
    ax.axvline(TRUE_SIN, color='black', lw=2, ls='--', label=f'True = {TRUE_SIN:.4f}')
    ax.axvline(arr.mean(), color=color, lw=2, label=f'Mean = {arr.mean():.4f}')
    ax.set_title(f'{method}\nVar = {np.var(arr):.2e}')
    ax.set_xlabel('Estimate')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle(f'MC Variance Reduction: N={N_COMP} samples, {N_REPS} experiments',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mc_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nKey Takeaways:')
print('- All three methods are unbiased (means close to true value)')
print('- Variance reduction translates directly to fewer samples needed')
print('- Control variates typically outperform antithetic variates when a good CV exists')
print('- Importance sampling (not shown here) is most powerful for tail/rare-event problems')